# 02 — NLP Feature Extraction
Two approaches compared:
- **Approach A:** TF-IDF bag-of-words features from job descriptions
- **Approach B:** LLM-extracted structured fields (seniority, skills, domain) via prompt engineering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/processed/job_postings_clean.csv')
df_desc = pd.read_csv('../data/raw/job_descriptions.csv')  # ravindrasinghrana dataset
print(f'Postings: {df.shape}')
print(f'Descriptions: {df_desc.shape}')
df_desc.head(2)

## 2. Text Preprocessing

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)       # remove HTML tags
    text = re.sub(r'[^a-z\s]', ' ', text)      # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()   # normalize whitespace
    return text

df['description_clean'] = df['description'].apply(clean_text)
print('Sample cleaned text:')
print(df['description_clean'].iloc[0][:300])

## 3. Approach A — TF-IDF Features

In [ ]:
tfidf = TfidfVectorizer(
    max_features=200,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=5
)

tfidf_matrix = tfidf.fit_transform(df['description_clean'])
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')

# Reduce to 50 dimensions with SVD (LSA)
svd = TruncatedSVD(n_components=50, random_state=42)
tfidf_reduced = svd.fit_transform(tfidf_matrix)
print(f'Reduced shape: {tfidf_reduced.shape}')
print(f'Variance explained: {svd.explained_variance_ratio_.sum():.2%}')

In [ ]:
# Top keywords overall
feature_names = tfidf.get_feature_names_out()
mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
top_idx = mean_tfidf.argsort()[-20:][::-1]
print('Top 20 TF-IDF terms:')
for i in top_idx:
    print(f'  {feature_names[i]}: {mean_tfidf[i]:.4f}')

In [ ]:
# Top keywords per salary band
bands = df['salary_band'].dropna().unique()
fig, axes = plt.subplots(1, len(bands), figsize=(16, 4))

for ax, band in zip(axes, sorted(bands)):
    mask = df['salary_band'] == band
    band_matrix = tfidf_matrix[mask]
    band_means = np.asarray(band_matrix.mean(axis=0)).flatten()
    top = band_means.argsort()[-10:][::-1]
    ax.barh([feature_names[i] for i in top], band_means[top], color='steelblue')
    ax.set_title(f'{band}')
    ax.invert_yaxis()

plt.suptitle('Top TF-IDF Terms per Salary Band', y=1.02)
plt.tight_layout()
plt.savefig('../docs/figures/04_tfidf_per_band.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Approach B — LLM Prompt Extraction
We use Claude API to extract structured fields from job descriptions: required skills, domain, and implied seniority. This is compared to TF-IDF in modeling notebook.

In [ ]:
import anthropic
import json

client = anthropic.Anthropic()  # uses ANTHROPIC_API_KEY env var

PROMPT_TEMPLATE = """You are a structured data extractor. Given a job description, extract the following fields.
Respond ONLY with valid JSON, no explanation.

Fields:
- domain: one of [tech, finance, healthcare, marketing, operations, education, other]
- seniority_implied: one of [junior, mid, senior, lead, unknown]
- top_skills: list of up to 5 key skills mentioned
- remote_friendly: true or false

Job description:
{description}
"""

def extract_features_llm(description, max_chars=1500):
    desc = str(description)[:max_chars]
    try:
        response = client.messages.create(
            model='claude-sonnet-4-20250514',
            max_tokens=300,
            messages=[{'role': 'user', 'content': PROMPT_TEMPLATE.format(description=desc)}]
        )
        text = response.content[0].text.strip()
        return json.loads(text)
    except Exception as e:
        return {'domain': 'other', 'seniority_implied': 'unknown', 'top_skills': [], 'remote_friendly': False}

# Test on one example
sample = df['description'].iloc[0]
result = extract_features_llm(sample)
print('Sample LLM extraction:')
print(json.dumps(result, indent=2))

In [ ]:
# Run on a sample of 500 rows (to save API cost)
# For full run, remove the .head(500)
SAMPLE_SIZE = 500
df_sample = df.head(SAMPLE_SIZE).copy()

print(f'Extracting LLM features for {SAMPLE_SIZE} rows...')
llm_features = []
for i, row in df_sample.iterrows():
    feats = extract_features_llm(row['description'])
    llm_features.append(feats)
    if (len(llm_features)) % 50 == 0:
        print(f'  {len(llm_features)}/{SAMPLE_SIZE} done')

df_llm = pd.DataFrame(llm_features)
df_llm.index = df_sample.index
df_llm.head()

In [ ]:
# Merge and save
df_with_llm = df_sample.join(df_llm)
df_with_llm.to_csv('../data/processed/job_postings_with_llm_features.csv', index=False)
print(f'Saved: data/processed/job_postings_with_llm_features.csv')

# Save TF-IDF vectorizer and SVD for later use
with open('../src/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
with open('../src/svd.pkl', 'wb') as f:
    pickle.dump(svd, f)

print('Saved: src/tfidf_vectorizer.pkl, src/svd.pkl')

## 5. Prompt Comparison
We test two prompting strategies and compare output quality.

In [ ]:
PROMPT_V2 = """Analyze this job posting and return JSON with these exact keys:
domain, seniority_implied, top_skills (array), remote_friendly (bool).

Use these domain values: tech, finance, healthcare, marketing, operations, education, other
Use these seniority values: junior, mid, senior, lead, unknown

Be concise. JSON only.

Posting: {description}
"""

# Test both prompts on 10 samples and compare
test_samples = df['description'].head(10).tolist()
results_v1, results_v2 = [], []

for desc in test_samples:
    r1 = extract_features_llm(desc)
    results_v1.append(r1)
    try:
        resp = client.messages.create(
            model='claude-sonnet-4-20250514',
            max_tokens=300,
            messages=[{'role': 'user', 'content': PROMPT_V2.format(description=str(desc)[:1500])}]
        )
        results_v2.append(json.loads(resp.content[0].text.strip()))
    except:
        results_v2.append({'domain': 'other', 'seniority_implied': 'unknown', 'top_skills': [], 'remote_friendly': False})

# Compare domain agreement
agree = sum(r1['domain'] == r2['domain'] for r1, r2 in zip(results_v1, results_v2))
print(f'Domain agreement between prompts: {agree}/10 ({agree*10}%)')

# Save comparison
comparison = pd.DataFrame({
    'prompt_v1_domain': [r['domain'] for r in results_v1],
    'prompt_v2_domain': [r['domain'] for r in results_v2],
    'prompt_v1_seniority': [r['seniority_implied'] for r in results_v1],
    'prompt_v2_seniority': [r['seniority_implied'] for r in results_v2],
})
comparison.to_csv('../docs/nlp_prompt_comparison.csv', index=False)
print('Saved: docs/nlp_prompt_comparison.csv')
comparison